# CNN_5 จำลองตามบทความ (4 ช่อง, 100 Hz, Gaussian ต่อ pattern, SGD): Sleep Stage Classification (HMC)

## Section 1 Import และตั้งค่า

In [ ]:
!pip install -q mne

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import json
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.signal import resample_poly
from IPython.display import display
import mne
from matplotlib.lines import Line2D
from matplotlib.ticker import MultipleLocator

import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    precision_recall_fscore_support,
    cohen_kappa_score,
)

PROJECT_DIR = Path("/content/drive/MyDrive/SleepStageClassificationProject")
DATA_DIR = PROJECT_DIR / "dataset" / "preprocess_microvolt"
RESULT_DIR = PROJECT_DIR / "results" / "CNN_5_gaussian"
RESULT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_PATH = RESULT_DIR / "cnn5_replica_best_model.keras"

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)


FS_RAW = 256
FS = 100
EPOCH_LEN_RAW = 30 * FS_RAW
EPOCH_LEN = 30 * FS
ALL_CHANNELS = ["F4-M1", "C4-M1", "O2-M1", "C3-M2", "E1-M2", "E2-M2", "Chin"]
SELECTED_CHANNELS = ["C4-M1", "C3-M2", "E1-M2", "Chin"]
CHANNEL_IDX = [ALL_CHANNELS.index(c) for c in SELECTED_CHANNELS]
N_ALL_CHANNELS = len(ALL_CHANNELS)
N_CHANNELS = len(SELECTED_CHANNELS)
SEQ_LEN = 5
STAGES = ["W", "N1", "N2", "N3", "R"]
LABELS = list(range(len(STAGES)))
LABEL_MAP = {stage: i for i, stage in enumerate(STAGES)}
SPLITS = ["train", "val", "test"]
INPUT_SHAPE = (SEQ_LEN * EPOCH_LEN, N_CHANNELS)
CNN_FILTERS = [8, 16, 32]
KERNEL_SIZE = 100
DENSE_UNITS = 50
DROPOUT = 0.5

BATCH_SIZE = 100
MAX_TRAIN_EPOCHS = 30
EVALS_PER_EPOCH = 5
LEARNING_RATE = 0.001
LR_DECAY_EVERY = 10
MIN_LR = 1e-6
PATIENCE = 10

## Section 2 โหลดข้อมูล เลือก 4 ช่อง resample เป็น 100 Hz และทำ Gaussian Standardization

In [ ]:
split_subjects = {
    split: sorted(
        path.stem
        for path in (DATA_DIR / split).glob("*.npz")
    )
    for split in SPLITS
}

assert not set(split_subjects["train"]) & set(split_subjects["val"])
assert not set(split_subjects["train"]) & set(split_subjects["test"])
assert not set(split_subjects["val"]) & set(split_subjects["test"])

for split in SPLITS:
    print(f"{split}: {len(split_subjects[split])} subjects")

def standardize_pattern(x):
    mean = x.mean(axis=0, keepdims=True)
    std = x.std(axis=0, keepdims=True)
    return ((x - mean) / (std + 1e-8)).astype(np.float32)

def flatten_values(value):
    if isinstance(value, np.ndarray):
        if value.ndim == 0:
            yield value.item()
        else:
            for item in value:
                yield from flatten_values(item)
    elif isinstance(value, (list, tuple)):
        for item in value:
            yield from flatten_values(item)
    else:
        yield value

def load_sequences(split, subject, with_x=True):
    path = DATA_DIR / split / f"{subject}.npz"
    if not path.exists():
        raise FileNotFoundError(f"ไม่พบไฟล์: {path}")

    X_list, y_list, ep_list, stage_list = [], [], [], []

    with np.load(path, allow_pickle=True) as data:
        for g in range(len(data["stage"])):
            stage = str(data["stage"][g])

            if stage not in LABEL_MAP:
                continue
            if int(data["n_valid"][g]) != SEQ_LEN:
                continue

            y_group = np.asarray(list(flatten_values(data["y"][g]))).astype(str)
            ep_group = np.asarray(list(flatten_values(data["epoch_indices"][g])),dtype=int)

            if len(y_group) != SEQ_LEN or len(ep_group) != SEQ_LEN:
                continue
            if not np.all(y_group.astype(str) == stage):
                continue
            if not np.all(np.diff(ep_group) == 1):
                continue

            if with_x:
                x_group = np.asarray(data["X"][g], dtype=np.float32)

                if x_group.shape != (SEQ_LEN, EPOCH_LEN_RAW, N_ALL_CHANNELS):
                    continue

                if not np.isfinite(x_group).all():
                    continue

                x_group = x_group[:, :, CHANNEL_IDX]
                x_group = x_group.reshape(SEQ_LEN * EPOCH_LEN_RAW, N_CHANNELS)
                x_group = resample_poly(x_group, FS, FS_RAW, axis=0)
                x_group = standardize_pattern(x_group)
                X_list.append(x_group)

            y_list.append(LABEL_MAP[stage])
            ep_list.append(int(ep_group[-1]))
            stage_list.append(stage)

    if with_x:
        X = (
            np.stack(X_list).astype(np.float32)
            if X_list
            else np.empty((0,) + INPUT_SHAPE, dtype=np.float32)
        )
    else:
        X = None

    return X, np.asarray(y_list, dtype=np.int32), np.asarray(ep_list, dtype=int), stage_list

rows = []

for split in SPLITS:
    for subject in split_subjects[split]:
        _, y, _, stage_list = load_sequences(
            split, subject, with_x=False
        )

        stage_counts = {
            stage: stage_list.count(stage)
            for stage in STAGES
        }

        rows.append({
            "split": split,
            "subject": subject,
            **stage_counts,
            "n_valid_sequences": len(y),
        })

subject_counts = pd.DataFrame(rows)

subject_counts.to_csv(
    RESULT_DIR / "cnn5_subject_stage_counts.csv",
    index=False,
)

is_complete = (subject_counts[STAGES] == 1).all(axis=1)

print("Subject ที่ไม่ผ่านเกณฑ์")
display(subject_counts[~is_complete])

selected_counts = subject_counts[is_complete].reset_index(drop=True)

split_subjects = {
    split: selected_counts.loc[
        selected_counts["split"] == split, "subject"
    ].tolist()
    for split in SPLITS
}

assert not set(split_subjects["train"]) & set(split_subjects["val"])
assert not set(split_subjects["train"]) & set(split_subjects["test"])
assert not set(split_subjects["val"]) & set(split_subjects["test"])

actual_counts = {
    split: len(split_subjects[split])
    for split in SPLITS
}

print("จำนวน Subject ที่ผ่านเกณฑ์:", actual_counts)

assert actual_counts == {
    "train": 79,
    "val": 21,
    "test": 23,
}, f"จำนวน Subject ไม่ตรงกับที่คาดไว้: {actual_counts}"


def load_split(split):
    Xs, ys = [], []

    for subject in split_subjects[split]:
        X, y, _, _ = load_sequences(split, subject, with_x=True)

        if len(y) == 0:
            continue

        Xs.append(X)
        ys.append(y)

    if not Xs:
        raise ValueError(f"No valid sequences found for {split}")

    return np.concatenate(Xs), np.concatenate(ys)


def make_dataset_from_arrays(X, y, shuffle):
    def generator():
        idx = np.random.permutation(len(y)) if shuffle else np.arange(len(y))
        for i in idx:
            yield X[i], y[i]

    ds = tf.data.Dataset.from_generator(
        generator,
        output_signature=(
            tf.TensorSpec(shape=INPUT_SHAPE, dtype=tf.float32),
            tf.TensorSpec(shape=(), dtype=tf.int32),
        ),
    )

    return ds.batch(BATCH_SIZE).prefetch(1)


X_train, y_train = load_split("train")
X_val, y_val = load_split("val")

print("Train:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)

print("ช่องที่ใช้:", SELECTED_CHANNELS)
print("Mean ต่อช่อง (train):", X_train.mean(axis=(0, 1)).round(4))
print("SD ต่อช่อง (train):", X_train.std(axis=(0, 1)).round(4))

train_ds = make_dataset_from_arrays(X_train, y_train, shuffle=True).repeat()
val_ds = make_dataset_from_arrays(X_val, y_val, shuffle=False)
STEPS_PER_EVAL = int(np.ceil(len(y_train) / BATCH_SIZE / EVALS_PER_EPOCH))

In [ ]:
print("Train:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Mean ต่อช่อง:", X_train.mean(axis=(0, 1)).round(4))
print("SD ต่อช่อง:", X_train.std(axis=(0, 1)).round(4))

## Section 3 สร้าง CNN_5

In [ ]:
def build_cnn5():
    inputs = layers.Input(shape=INPUT_SHAPE)
    x = layers.Permute((2, 1))(inputs)
    x = layers.Reshape((N_CHANNELS, SEQ_LEN * EPOCH_LEN, 1))(x)

    for n_filters in CNN_FILTERS:
        x = layers.Conv2D(
            filters=n_filters,
            kernel_size=(1, KERNEL_SIZE),
            strides=(1, 1),
            padding="same",
        )(x)
        x = layers.ReLU()(x)
        x = layers.BatchNormalization()(x)
        x = layers.AveragePooling2D(
            pool_size=(1, 2),
            strides=(1, 2),
        )(x)

    x = layers.Flatten()(x)
    x = layers.Dense(DENSE_UNITS, activation="relu")(x)
    x = layers.Dropout(DROPOUT)(x)
    outputs = layers.Dense(len(STAGES), activation="softmax")(x)

    return models.Model(inputs, outputs)


model = build_cnn5()
model.compile(
    optimizer=tf.keras.optimizers.SGD(learning_rate=LEARNING_RATE),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
model.summary()

## Section 4 Train Model

In [ ]:
def lr_schedule(step, lr):
    train_epoch = step // EVALS_PER_EPOCH
    decay_count = train_epoch // LR_DECAY_EVERY
    return max(LEARNING_RATE * (0.1 ** decay_count), MIN_LR)


callbacks = [
    tf.keras.callbacks.LearningRateScheduler(lr_schedule),
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(MODEL_PATH),
        monitor="val_loss",
        save_best_only=True,
        mode="min",
        verbose=1,
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=PATIENCE,
        restore_best_weights=True,
        mode="min",
        verbose=1,
    ),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=MAX_TRAIN_EPOCHS * EVALS_PER_EPOCH,
    steps_per_epoch=STEPS_PER_EVAL,
    callbacks=callbacks,
)

history_df = pd.DataFrame(history.history)
history_df.insert(0, "epoch", range(1, len(history_df) + 1))
history_df.insert(1, "train_epoch", history_df["epoch"] / EVALS_PER_EPOCH)
history_df = history_df.rename(columns={"loss": "train_loss", "accuracy": "train_accuracy"})
history_df.to_csv(RESULT_DIR / "cnn5_training_history.csv", index=False)

display(history_df)

for metric, title in [("loss", "Loss"), ("accuracy", "Accuracy")]:
    plt.figure(figsize=(8, 5))
    plt.plot(history_df["epoch"], history_df[f"train_{metric}"], label=f"Train {title}")
    plt.plot(history_df["epoch"], history_df[f"val_{metric}"], label=f"Validation {title}")
    plt.xlabel("Evaluation step (1/5 training epoch)")
    plt.ylabel(title)
    plt.title(f"Train / Validation {title}")
    plt.legend()
    plt.grid(True)
    plt.show()

## Section 5 ประเมินผลด้วย Validation Set และ Test Set

In [ ]:
best_model = tf.keras.models.load_model(MODEL_PATH)


def evaluate(split):
    subject_ids, target_epochs, y_true, y_pred = [], [], [], []

    for subject in split_subjects[split]:
        X, y, target_epoch, _ = load_sequences(split, subject)
        prob = best_model.predict(X,batch_size=BATCH_SIZE,verbose=0)

        subject_ids.extend([subject] * len(y))
        target_epochs.extend(target_epoch.tolist())
        y_true.extend(y.tolist())
        y_pred.extend(np.argmax(prob, axis=1).tolist())

    y_true = np.asarray(y_true, dtype=np.int32)
    y_pred = np.asarray(y_pred, dtype=np.int32)

    assert len(y_true) > 0

    accuracy = float(np.mean(y_true == y_pred))
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true, y_pred, labels=LABELS, zero_division=0
    )
    macro_precision = float(precision.mean())
    macro_recall = float(recall.mean())
    macro_f1 = float(f1.mean())
    cohen_kappa = float(cohen_kappa_score(y_true, y_pred, labels=LABELS))

    print(f"{split.upper()} sequences:", len(y_true))
    print("Accuracy:", round(accuracy, 4))
    print("Macro Precision:", round(macro_precision, 4))
    print("Macro Recall:", round(macro_recall, 4))
    print("Macro F1-score:", round(macro_f1, 4))
    print("Cohen's Kappa:", round(cohen_kappa, 4))
    print(classification_report(y_true, y_pred, labels=LABELS, target_names=STAGES, digits=4, zero_division=0))

    metrics_df = pd.DataFrame({
        "class": STAGES,
        "precision": precision,
        "recall": recall,
        "f1_score": f1,
        "support": support,
    })

    metrics_df = pd.concat([
        metrics_df,
        pd.DataFrame([
            {"class": "macro avg", "precision": macro_precision, "recall": macro_recall,
             "f1_score": macro_f1, "support": int(support.sum())},
            {"class": "accuracy", "precision": accuracy, "recall": accuracy,
             "f1_score": accuracy, "support": int(support.sum())},
            {"class": "cohen_kappa", "precision": np.nan, "recall": np.nan,
             "f1_score": cohen_kappa, "support": int(support.sum())},
        ]),
    ], ignore_index=True)

    metrics_df.to_csv(RESULT_DIR / f"cnn5_{split}_metrics.csv", index=False)
    display(metrics_df)

    cm = confusion_matrix(y_true, y_pred, labels=LABELS)
    cm_norm = confusion_matrix(y_true, y_pred, labels=LABELS, normalize="true")
    pd.DataFrame(cm, index=STAGES, columns=STAGES).to_csv(RESULT_DIR / f"cnn5_{split}_confusion_matrix.csv")

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    ConfusionMatrixDisplay(cm, display_labels=STAGES).plot(ax=axes[0], cmap="Blues", colorbar=False)
    axes[0].set_title(f"{split} Confusion Matrix (count)")
    ConfusionMatrixDisplay(cm_norm, display_labels=STAGES).plot(
        ax=axes[1], cmap="Blues", values_format=".2f", colorbar=False
    )
    axes[1].set_title(f"{split} Confusion Matrix (normalized by actual)")
    plt.tight_layout()
    plt.savefig(RESULT_DIR / f"cnn5_{split}_confusion_matrix.png", dpi=150)
    plt.show()

    pred_df = pd.DataFrame({
        "subject_id": subject_ids,
        "target_epoch": target_epochs,
        "actual": [STAGES[i] for i in y_true],
        "predicted": [STAGES[i] for i in y_pred],
    })
    pred_df["correct"] = pred_df["actual"] == pred_df["predicted"]
    pred_df.to_csv(RESULT_DIR / f"cnn5_{split}_predictions.csv", index=False)

    return {
        "accuracy": accuracy,
        "macro_precision": macro_precision,
        "macro_recall": macro_recall,
        "macro_f1": macro_f1,
        "cohen_kappa": cohen_kappa,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "support": support,
        "pred_df": pred_df,
    }


val_result = evaluate("val")
test_result = evaluate("test")

In [ ]:
def kappa_of(X, y):
    pred = best_model.predict(X, batch_size=BATCH_SIZE, verbose=0).argmax(1)
    return cohen_kappa_score(y, pred, labels=LABELS)

X_test, y_test = load_split("test")

kappa_table = pd.DataFrame([{
    "model": "CNN_5",
    "train_epochs_run": len(history_df),
    "TR": kappa_of(X_train, y_train),
    "VAL": kappa_of(X_val, y_val),
    "TS": kappa_of(X_test, y_test),
}]).round(3)

kappa_table.to_csv(RESULT_DIR / "cnn5_kappa_TR_VAL_TS.csv", index=False)
display(kappa_table)

## Section 6 Actual vs Predicted (Test Set)

In [ ]:

pred_df = test_result["pred_df"]

print("จำนวน test sequences:", len(pred_df))
print("ทำนายถูก:", int(pred_df["correct"].sum()))
print("ทำนายผิด:", int((~pred_df["correct"]).sum()))

display(pred_df.head(30))

reference_rows = []

for subject_id in split_subjects["test"]:
    _, y, target_epochs, _ = load_sequences(
        "test", subject_id, with_x=False
    )

    reference_rows.extend([
        {
            "subject_id": subject_id,
            "target_epoch": int(epoch),
            "actual": STAGES[int(label)],
        }
        for label, epoch in zip(y, target_epochs)
    ])

reference = pd.DataFrame(
    reference_rows,
    columns=["subject_id", "target_epoch", "actual"]
)

key = ["subject_id", "target_epoch", "actual"]

reference = reference[key].sort_values(key).reset_index(drop=True)
current = pred_df[key].sort_values(key).reset_index(drop=True)

assert reference.equals(current), (
    "ชุด Test ไม่ตรงกับข้อมูลใน preprocess_microvolt"
)

print("ชุด Test ตรงกับข้อมูลใน preprocess_microvolt ทุกตัวอย่าง")

## Section 7 สรุปผล

In [ ]:
best_row = history_df.loc[history_df["val_loss"].idxmin()]

print("Best step (เลือกจาก val_loss):", int(best_row["epoch"]), "| training epoch:", round(best_row["train_epoch"], 2))
print("Train Loss:", round(best_row["train_loss"], 4), "| Train Accuracy:", round(best_row["train_accuracy"], 4))
print("Val Loss:", round(best_row["val_loss"], 4), "| Val Accuracy:", round(best_row["val_accuracy"], 4))
print()

for name, r in [("Validation", val_result), ("Test", test_result)]:
    print(name)
    print("Accuracy:", round(r["accuracy"], 4))
    print("Macro Precision:", round(r["macro_precision"], 4))
    print("Macro Recall:", round(r["macro_recall"], 4))
    print("Macro F1-score:", round(r["macro_f1"], 4))
    print("Cohen's Kappa:", round(r["cohen_kappa"], 4))
    for i, stage in enumerate(STAGES):
        print(f"{stage}: Precision {r['precision'][i]:.4f} | Recall {r['recall'][i]:.4f} | F1 {r['f1'][i]:.4f} | Support {r['support'][i]}")

## Section 8 Hypnogram เทียบระยะจริงกับที่โมเดลทำนาย (Test)

In [ ]:
pred_df = pd.read_csv(RESULT_DIR / "cnn5_test_predictions.csv")

test_table = pred_df.rename(columns={"subject_id": "subject"}).copy()
test_table["epoch_end"] = test_table["target_epoch"]
test_table["epoch_start"] = test_table["target_epoch"] - (SEQ_LEN - 1)
tables = {"test": test_table}

X_test, y_test = load_split("test")
assert (y_test == test_table["actual"].map(LABEL_MAP).to_numpy()).all(), "ลำดับ X_test ไม่ตรงกับ pred_df"

DATA = {"test": {"X": X_test, "y": y_test, "meta": test_table}}
print("test_table:", test_table.shape, "| X_test:", X_test.shape)

In [ ]:
print(f"Accuracy: {test_table['correct'].mean():.4f}")
print(f"Correct: {test_table['correct'].sum()}/{len(test_table)}")

stage_y = {"W": 4, "R": 3, "N1": 2, "N2": 1, "N3": 0}

hyp_dir = RESULT_DIR / "hypnogram"
hyp_dir.mkdir(parents=True, exist_ok=True)

FULL_NIGHT_DIR = PROJECT_DIR / "dataset" / "Test_Full_night"
EPOCH_SEC = 30

FIG_W, FIG_H = 30, 10
GT_COLOR = "#1f5fff"
PRED_LINE_COLOR = "black"
LINE_W = 1.5
BLOCK_W = 8
MARKER_SIZE = 90
PRED_MARKER_SIZE = 40
DASH = (0, (5, 3))

ann_to_stage = {
    "sleep stage w": "W",
    "sleep stage n1": "N1", "sleep stage 1": "N1",
    "sleep stage n2": "N2", "sleep stage 2": "N2",
    "sleep stage n3": "N3", "sleep stage 3": "N3",
    "sleep stage r": "R",
}


def load_full_night(subject):
    ann = mne.read_annotations(str(FULL_NIGHT_DIR / f"{subject}_sleepscoring.edf"))

    items = []
    for onset, dur, desc in zip(ann.onset, ann.duration, ann.description):
        stage = ann_to_stage.get(str(desc).strip().lower())
        if stage is None:
            continue
        items.append((int(round(onset / EPOCH_SEC)),
                      max(1, int(round(dur / EPOCH_SEC))),
                      stage_y[stage]))

    levels = np.full(max(s + n for s, n, _ in items), np.nan)
    for s, n, level in items:
        levels[s:s + n] = level
    return levels


def best_offset(levels, sub, candidates=range(-3, 4)):
    best = (0, -1.0)
    for off in candidates:
        same = total = 0
        for r in sub.itertuples():
            for e in range(int(r.epoch_start), int(r.epoch_end) + 1):
                if 0 <= e + off < len(levels):
                    total += 1
                    same += int(levels[e + off] == stage_y[r.actual])
        agree = same / total if total else 0.0
        if agree > best[1]:
            best = (off, agree)
    return best


def shift(levels, off):
    if off > 0:
        return levels[off:]
    if off < 0:
        return np.concatenate([np.full(-off, np.nan), levels])
    return levels


def fill_gaps(values):
    filled = values.copy()
    valid = np.where(~np.isnan(filled))[0]
    if len(valid) == 0:
        return filled
    filled[:valid[0]] = filled[valid[0]]
    for i in range(valid[0] + 1, len(filled)):
        if np.isnan(filled[i]):
            filled[i] = filled[i - 1]
    return filled


def build_three_lines(subject, sub):
    full_gt = load_full_night(subject)
    off, agree = best_offset(full_gt, sub)
    full_gt = shift(full_gt, off)
    n_sel_seq = len(sub)
    n_sel_ep = int((sub["epoch_end"] - sub["epoch_start"] + 1).sum())
    print(f"{subject}: epochs = {n_sel_ep} | sequence = {n_sel_seq}")

    n_ep = max(len(full_gt), int(sub["epoch_end"].max()) + 1)
    full_gt = np.concatenate([full_gt, np.full(n_ep - len(full_gt), np.nan)])

    sel_gt = np.full(n_ep, np.nan)
    pred = np.full(n_ep, np.nan)
    ok = np.full(n_ep, -1, dtype=int)

    for r in sub.itertuples():
        sl = slice(int(r.epoch_start), int(r.epoch_end) + 1)
        sel_gt[sl] = stage_y[r.actual]
        pred[sl] = stage_y[r.predicted]
        ok[sl] = int(r.correct)
    return full_gt, sel_gt, pred, ok


for subject in test_table["subject"].unique():
    sub = test_table[test_table["subject"] == subject].sort_values("epoch_start").copy()
    full_gt, sel_gt, pred, ok = build_three_lines(subject, sub)
    n_ep = len(full_gt)

    x_start = np.arange(n_ep) * EPOCH_SEC / 3600
    x_end = (np.arange(n_ep) + 1) * EPOCH_SEC / 3600
    x_step = np.append(x_start, x_end[-1])
    full_step = np.append(full_gt, full_gt[-1])

    cx = (sub["epoch_start"].to_numpy() + sub["epoch_end"].to_numpy() + 1) / 2 * EPOCH_SEC / 3600
    sx0 = sub["epoch_start"].to_numpy() * EPOCH_SEC / 3600
    sx1 = (sub["epoch_end"].to_numpy() + 1) * EPOCH_SEC / 3600
    gy = sub["actual"].map(stage_y).to_numpy()
    py = sub["predicted"].map(stage_y).to_numpy()
    correct = sub["correct"].to_numpy()
    ok_color = np.where(correct, "green", "red")

    fig, axes = plt.subplots(3, 1, figsize=(FIG_W, FIG_H), sharex=True)

    axes[0].step(x_step, full_step, where="post", color="black", linewidth=LINE_W)

    axes[1].step(x_step, full_step, where="post", color="#9e9e9e", linewidth=LINE_W, zorder=1)
    axes[1].hlines(gy, sx0, sx1, color=GT_COLOR, linewidth=BLOCK_W, zorder=2)
    axes[1].scatter(cx, gy, s=MARKER_SIZE, color=GT_COLOR,
                    edgecolor="white", linewidth=1, zorder=3)

    axes[2].step(x_step, full_step, where="post", color="#9e9e9e", linewidth=LINE_W, zorder=1)
    axes[2].hlines(py, sx0, sx1, color=ok_color, linewidth=BLOCK_W, zorder=2)
    axes[2].scatter(cx, py, s=MARKER_SIZE, color=ok_color,
                    edgecolor="white", linewidth=1, zorder=3)

    for ax in axes[1:]:
        for x in cx:
            ax.axvline(x, color=GT_COLOR, linestyle="--", linewidth=0.8, alpha=0.4, zorder=0)

    axes[0].set_title(f"Real all night | {subject}")
    axes[1].set_title(f"Ground Truth (sequences used) | {subject}")
    axes[2].set_title(
        f"Predicted | {subject} | Correct {int(correct.sum())}/{len(sub)} "
        f"({correct.mean():.1%})"
    )

    for ax in axes:
        ax.set_ylabel("Sleep stage")
        ax.set_yticks(list(stage_y.values()))
        ax.set_yticklabels(list(stage_y.keys()))
        ax.set_ylim(-0.5, 4.5)
        ax.xaxis.set_major_locator(MultipleLocator(0.5))
        ax.grid(True, alpha=0.3)

    axes[0].set_xlim(-0.05, x_end[-1] * 1.12)

    axes[1].legend(handles=[
        Line2D([0], [0], color="#9e9e9e", lw=LINE_W, label="Real all night"),
        Line2D([0], [0], color=GT_COLOR, lw=4, label="Ground Truth (selected sequence)"),
    ], loc="lower right", framealpha=0.95)
    axes[2].set_xlabel("Time since recording start (hours)")
    axes[2].legend(handles=[
        Line2D([0], [0], color="#9e9e9e", lw=LINE_W, label="Real all night"),
        Line2D([0], [0], color="green", lw=4, label="Predicted: Correct"),
        Line2D([0], [0], color="red", lw=4, label="Predicted: Incorrect"),
    ], loc="lower right", framealpha=0.95)

    plt.tight_layout()
    plt.savefig(hyp_dir / f"hypnogram_full_{subject}.png", dpi=150, bbox_inches="tight")
    plt.show()
    plt.close()

## Section 8.1 สัญญาณดิบทั้ง subject เทียบระยะจริงกับที่โมเดลทำนาย (Test, แยกรายคน)

In [ ]:
PLOT_CHANNELS = SELECTED_CHANNELS
BIN = 1

def channel_type(name):
    if name == "Chin":
        return "EMG"
    if name[0] == "E" and name[1].isdigit():
        return "EOG"
    return "EEG"

stage_color = {"W": "#ffd6a5", "N1": "#fdffb6", "N2": "#a0c4ff", "N3": "#bdb2ff", "R": "#ffc6ff"}

test_subject_list = [
    "SN010", "SN013", "SN020", "SN021", "SN024", "SN028", "SN031", "SN038",
    "SN047", "SN058", "SN067", "SN079", "SN084", "SN085", "SN087", "SN101",
    "SN114", "SN117", "SN136", "SN137", "SN141", "SN146", "SN147",
]
available = set(test_table["subject"])
plot_subjects = [s for s in test_subject_list if s in available]
print(f"วาด {len(plot_subjects)} subjects จาก Test set (BIN={BIN})")
signal_dir = RESULT_DIR / "signal_whole_recording"
signal_dir.mkdir(parents=True, exist_ok=True)

X_all = DATA["test"]["X"]
ch_idx = [SELECTED_CHANNELS.index(c) for c in PLOT_CHANNELS]
n_sig = len(ch_idx)
n_bin = X_all.shape[1] // BIN

for subject in plot_subjects:
    sub = test_table[test_table["subject"] == subject].sort_values("epoch_start")

    keep, last_end = [], -1
    for i, r in sub.iterrows():
        if r["epoch_start"] > last_end:
            keep.append(i)
            last_end = r["epoch_end"]
    sub = sub.loc[keep]

    xs, los, his = [], [], []
    for i, r in sub.iterrows():
        t0 = r["epoch_start"] * 30 / 3600
        seg = np.asarray(X_all[i][: n_bin * BIN, :])[:, ch_idx].reshape(n_bin, BIN, n_sig)
        xs.append(np.append(t0 + (np.arange(n_bin) + 0.5) * BIN / FS / 3600, np.nan))
        los.append(np.vstack([seg.min(axis=1), np.full((1, n_sig), np.nan)]))
        his.append(np.vstack([seg.max(axis=1), np.full((1, n_sig), np.nan)]))
    x = np.concatenate(xs)
    lo = np.concatenate(los)
    hi = np.concatenate(his)

    fig, axes = plt.subplots(
        n_sig + 3, 1, figsize=(20, 1.2 * n_sig + 3.5), sharex=True,
        gridspec_kw={"height_ratios": [2] * n_sig + [1, 1, 0.6]}
    )

    for k in range(n_sig):
        axes[k].plot(x, hi[:, k], color="blue", linewidth=0.3)
        lim = np.nanpercentile(np.abs(np.concatenate([lo[:, k], hi[:, k]])), 99.5)
        axes[k].set_ylim(-lim, lim)
        axes[k].set_ylabel(f"{PLOT_CHANNELS[k]}\n[{channel_type(PLOT_CHANNELS[k])}] microvolt", fontsize=8)
        axes[k].grid(alpha=0.3)

    for i, r in sub.iterrows():
        t0 = r["epoch_start"] * 30 / 3600
        t1 = (r["epoch_end"] + 1) * 30 / 3600
        axes[-3].axvspan(t0, t1, facecolor=stage_color[r["actual"]], linewidth=0)
        axes[-2].axvspan(t0, t1, facecolor=stage_color[r["predicted"]], linewidth=0)
        axes[-1].axvspan(t0, t1, facecolor="green" if r["correct"] else "red", linewidth=0)
        for k in range(n_sig):
            axes[k].axvspan(t0, t1, facecolor="green" if r["correct"] else "red",
                            alpha=0.1, linewidth=0, zorder=0)

    for ax, name in [(axes[-3], "Ground Truth"), (axes[-2], "Predicted"), (axes[-1], "Correct?")]:
        ax.set_yticks([])
        ax.set_ylabel(name, rotation=0, ha="right", va="center")

    axes[-1].set_xlim(sub["epoch_start"].min() * 30 / 3600, (sub["epoch_end"].max() + 1) * 30 / 3600)
    axes[-1].set_xlabel("Time since recording start (hours)")

    axes[0].legend(handles=[
        Line2D([0], [0], color=stage_color[s], lw=8, label=s) for s in STAGES
    ] + [
        Line2D([0], [0], color="green", lw=8, label="Correct"),
        Line2D([0], [0], color="red", lw=8, label="Wrong"),
    ], loc="upper right", ncol=7, fontsize=8)

    n_ok = int(sub["correct"].sum())
    fig.suptitle(
        f"{subject} | whole recording | correct {n_ok}/{len(sub)} sequences ({n_ok / len(sub):.1%}) "
        f"(blank = no sequence)"
    )
    plt.tight_layout()
    plt.savefig(signal_dir / f"signal_whole_{subject}.png", dpi=150, bbox_inches="tight")
    plt.show()
    plt.close()

## Section 8.2 สัญญาณของแต่ละ sequence เทียบระยะการนอนจริงกับที่โมเดลทำนาย (Test, แยกรายคน)

In [ ]:
PLOT_CHANNELS = SELECTED_CHANNELS
BIN = 1
FIG_W = 30
GAP = 0.25

test_subject_list = [
    "SN010", "SN013", "SN020", "SN021", "SN024", "SN028", "SN031", "SN038",
    "SN047", "SN058", "SN067", "SN079", "SN084", "SN085", "SN087", "SN101",
    "SN114", "SN117", "SN136", "SN137", "SN141", "SN146", "SN147",
]


def channel_type(name):
    if name == "Chin":
        return "EMG"
    if name[0] == "E" and name[1].isdigit():
        return "EOG"
    return "EEG"


stage_color = {"W": "#ffd6a5", "N1": "#fdffb6", "N2": "#a0c4ff", "N3": "#bdb2ff", "R": "#ffc6ff"}

available = set(test_table["subject"])
plot_subjects = [s for s in test_subject_list if s in available]
print(f"วาด {len(plot_subjects)} subjects จาก Test set (BIN={BIN})")
signal_dir = RESULT_DIR / "signal_sequences"
signal_dir.mkdir(parents=True, exist_ok=True)

X_all = DATA["test"]["X"]
ch_idx = [SELECTED_CHANNELS.index(c) for c in PLOT_CHANNELS]
n_sig = len(ch_idx)
n_bin = X_all.shape[1] // BIN
seq_sec = X_all.shape[1] / FS

for subject in plot_subjects:
    sub = test_table[test_table["subject"] == subject].sort_values("epoch_start")

    keep, last_end = [], -1
    for i, r in sub.iterrows():
        if r["epoch_start"] > last_end:
            keep.append(i)
            last_end = r["epoch_end"]
    sub = sub.loc[keep]
    rows = list(sub.iterrows())
    n_seq = len(rows)

    xs, los, his = [], [], []
    for k, (i, r) in enumerate(rows):
        x0 = k * (1 + GAP)
        seg = np.asarray(X_all[i][: n_bin * BIN, :])[:, ch_idx].reshape(n_bin, BIN, n_sig)
        rel = (np.arange(n_bin) + 0.5) * BIN / FS / seq_sec
        xs.append(np.append(x0 + rel, np.nan))
        los.append(np.vstack([seg.min(axis=1), np.full((1, n_sig), np.nan)]))
        his.append(np.vstack([seg.max(axis=1), np.full((1, n_sig), np.nan)]))
    x = np.concatenate(xs)
    lo = np.concatenate(los)
    hi = np.concatenate(his)

    fig, axes = plt.subplots(
        n_sig + 3, 1, figsize=(FIG_W, 1.2 * n_sig + 3.5), sharex=True,
        gridspec_kw={"height_ratios": [2] * n_sig + [1, 1, 0.6]}
    )

    for k in range(n_sig):
        axes[k].plot(x, hi[:, k], color="blue", linewidth=0.3)
        lim = np.nanpercentile(np.abs(np.concatenate([lo[:, k], hi[:, k]])), 99.5)
        axes[k].set_ylim(-lim, lim)
        axes[k].set_ylabel(f"{PLOT_CHANNELS[k]}\n[{channel_type(PLOT_CHANNELS[k])}] microvolt", fontsize=8)
        axes[k].grid(alpha=0.3)

    centers, labels = [], []
    for k, (i, r) in enumerate(rows):
        t0, t1 = k * (1 + GAP), k * (1 + GAP) + 1
        axes[-3].axvspan(t0, t1, facecolor=stage_color[r["actual"]], linewidth=0)
        axes[-2].axvspan(t0, t1, facecolor=stage_color[r["predicted"]], linewidth=0)
        axes[-1].axvspan(t0, t1, facecolor="green" if r["correct"] else "red", linewidth=0)
        for kk in range(n_sig):
            axes[kk].axvspan(t0, t1, facecolor="green" if r["correct"] else "red",
                             alpha=0.1, linewidth=0, zorder=0)

        axes[-3].text((t0 + t1) / 2, 0.5, r["actual"], ha="center", va="center",
                      fontsize=10, fontweight="bold", transform=axes[-3].get_xaxis_transform())
        axes[-2].text((t0 + t1) / 2, 0.5, r["predicted"], ha="center", va="center",
                      fontsize=10, fontweight="bold", transform=axes[-2].get_xaxis_transform())
        centers.append((t0 + t1) / 2)
        labels.append(f"{r['epoch_start'] * 30 / 3600:.2f} h\n(epoch {int(r['epoch_start'])}-{int(r['epoch_end'])})")

    for k in range(n_seq - 1):
        xg = k * (1 + GAP) + 1 + GAP / 2
        for ax in axes:
            ax.axvline(xg, color="gray", linestyle=":", linewidth=1)

    for ax, name in [(axes[-3], "Ground Truth"), (axes[-2], "Predicted"), (axes[-1], "Correct?")]:
        ax.set_yticks([])
        ax.set_ylabel(name, rotation=0, ha="right", va="center")

    axes[-1].set_xlim(-GAP / 2, n_seq * (1 + GAP) - GAP / 2)
    axes[-1].set_xticks(centers)
    axes[-1].set_xticklabels(labels, fontsize=9)
    axes[-1].set_xlabel("Selected sequences in time order (gaps removed; tick = real time since recording start)")

    axes[0].legend(handles=[
        Line2D([0], [0], color=stage_color[s], lw=8, label=s) for s in STAGES
    ] + [
        Line2D([0], [0], color="green", lw=8, label="Correct"),
        Line2D([0], [0], color="red", lw=8, label="Wrong"),
    ], loc="lower right", bbox_to_anchor=(1, 1.01), ncol=7, fontsize=8)

    n_ok = int(sub["correct"].sum())
    fig.suptitle(
        f"{subject} | selected sequences | correct {n_ok}/{len(sub)} sequences ({n_ok / len(sub):.1%})"
    )
    plt.tight_layout()
    plt.savefig(signal_dir / f"signal_sequences_{subject}.png", dpi=150, bbox_inches="tight")
    plt.show()
    plt.close()